<img src="images/logo.png" width="15%" height="15%">

# **LangChain Essentials**

In [ ]:
%pip install yfinance langchain langchain-mistralai python-dotenv

### **6. TodoListMiddleware**

Short extract from the [Documentation](https://docs.langchain.com/oss/python/langchain/middleware/overview):

- The core agent loop involves calling a model, letting it choose tools to execute, and then finishing when it calls no more tools:

<img src="images/react_agent.png" width="20%" height="20%">

- **Middleware** exposes hooks before and after each of those steps:

<img src="images/middleware.png" width="25%" height="25%">

---

In the case of the [**TodoListMiddleware**](https://reference.langchain.com/python/langchain/agents/middleware/todo/TodoListMiddleware), your model has the ability to track tasks with simple dictionaries.

It can create them and update their status.

```python
    [
        {"content": "Do task 1", "status": "in_progress"},
        {"content": "Do task 2", "status": "completed"}
    ]
```

The updated plan is injected in the [system prompt](https://reference.langchain.com/python/langchain/agents/middleware/todo/WRITE_TODOS_SYSTEM_PROMPT) at each conversation turn, forcing your agent to take into account this context.

It can be extremely beneficial for long-horizon tasks, to better plan the overall task and avoid hallucination.

---

You can also check out my own implementation of a todo list middleware on [Github](https://github.com/tom4112/tinyharness): an improved version with task IDs and changes history.



In [11]:
import yfinance as yf
from typing import Optional, Dict, List
from langchain.tools import tool

# Tools created by the Mistral Vibe VS Code Extension

@tool
def get_financial_data(ticker: str, period: str = "1mo", interval: str = "1d") -> Optional[Dict]:
    """
    Retrieve financial data for a given ticker symbol.
    
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT')
        period: Data period - 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
        interval: Data interval - 1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo
    
    Returns:
        Dictionary containing financial data or None if fetch fails
    """
    try:
        stock = yf.Ticker(ticker)
        
        # Get historical market data
        hist = stock.history(period=period, interval=interval)
        
        # Get info dictionary
        info = stock.info
        
        # Get current price
        current_price = info.get('currentPrice', info.get('regularMarketPrice'))
        
        # Get additional metrics
        financial_data = {
            'ticker': ticker,
            'current_price': current_price,
            'currency': info.get('currency'),
            'market_cap': info.get('marketCap'),
            'pe_ratio': info.get('trailingPE'),
            'forward_pe': info.get('forwardPE'),
            'peg_ratio': info.get('pegRatio'),
            'dividend_yield': info.get('dividendYield'),
            'beta': info.get('beta'),
            '52_week_high': info.get('fiftyTwoWeekHigh'),
            '52_week_low': info.get('fiftyTwoWeekLow'),
            '50_day_ma': info.get('fiftyDayAverage'),
            '200_day_ma': info.get('twoHundredDayAverage'),
            'volume': info.get('volume'),
            'avg_volume': info.get('averageVolume'),
            'open': info.get('open'),
            'previous_close': info.get('regularMarketPreviousClose'),
            'day_high': info.get('dayHigh'),
            'day_low': info.get('dayLow'),
            'historical_data': hist.to_dict('index') if not hist.empty else {}
        }
        
        return financial_data
        
    except Exception as e:
        print(f"Error fetching financial data for {ticker}: {e}")
        return None

@tool
def get_ticker_news(ticker: str, limit: int = 10) -> Optional[List[Dict]]:
    """
    Retrieve news articles for a given ticker symbol.
    
    Note: yfinance news may be rate-limited or unavailable for some tickers.
    
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT')
        limit: Maximum number of news articles to return
    
    Returns:
        List of news articles (dictionaries) or None if fetch fails
    """
    try:
        stock = yf.Ticker(ticker)
        
        # Try different methods to get news - yfinance API has changed
        news_items = None
        
        # Method 1: Use .news property (older versions)
        if hasattr(stock, 'news'):
            news_items = stock.news
        
        # Method 2: Try .get_news() if available
        if not news_items and hasattr(stock, 'get_news'):
            news_items = stock.get_news()
        
        # Method 3: Use the actions or info which might contain news
        if not news_items:
            info = stock.info
            news_items = info.get('news', [])
        
        if not news_items:
            print(f"Warning: No news found for {ticker}. News may not be available via yfinance.")
            return []
        
        # Ensure news_items is a list
        if not isinstance(news_items, list):
            news_items = []
        
        # Process and return top 'limit' news articles
        processed_news = []
        for item in news_items[:limit]:
            # Handle different possible key names in yfinance news
            title = item.get('title') or item.get('headline') or item.get('title_text') or str(item)
            publisher = item.get('publisher') or item.get('source') or item.get('provider') or 'Unknown'
            link = item.get('link') or item.get('url') or item.get('news_url') or ''
            timestamp = item.get('providerPublishTime') or item.get('time') or item.get('published') or item.get('date') or 0
            summary = item.get('summary') or item.get('abstract') or item.get('description') or ''
            news_type = item.get('type') or item.get('news_type') or 'NEWS'
            related_tickers = item.get('relatedTickers') or item.get('related_tickers') or []
            
            processed_news.append({
                'title': title,
                'publisher': publisher,
                'link': link,
                'timestamp': timestamp,
                'type': news_type,
                'related_tickers': related_tickers,
                'summary': summary
            })
        
        return processed_news
        
    except Exception as e:
        print(f"Error fetching news for {ticker}: {e}")
        return None

toolbox = [get_financial_data, get_ticker_news]

In [12]:
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI

load_dotenv()
llm = ChatMistralAI(model="mistral-small-latest")
llm_with_tools = llm.bind_tools(toolbox)

In [13]:
from langchain.agents.middleware import TodoListMiddleware

agent = create_agent(
    model=llm_with_tools,
    tools=toolbox,
    system_prompt="You are a financial analyst using tools to research stocks",
    middleware=[TodoListMiddleware()]
)

In [14]:
from langchain_core.messages import HumanMessage

msg = HumanMessage(content="""
    How would you research the Nvidia (NVDA) stock? 
    Update your internal todo list to reflect each step which you would perform and use your tools to start the research.
    """
)

state = agent.invoke(
    {"messages": [msg]}
)

In [15]:
# In addition to messages, your state also contains a 'todos' key
state

{'messages': [HumanMessage(content='\n    How would you research the Nvidia (NVDA) stock? \n    Update your internal todo list to reflect each step which you would perform and use your tools to start the research.\n    ', additional_kwargs={}, response_metadata={}, id='07f255cf-33d2-414e-8c80-259742b7523f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3fot1yEpr', 'type': 'function', 'function': {'name': 'write_todos', 'arguments': '{"todos": [{"content": "Identify key metrics and data points to research for NVDA stock (e.g., historical prices, financials, news, valuation metrics).", "status": "in_progress"}, {"content": "Fetch historical financial data for NVDA using yfinance.", "status": "pending"}, {"content": "Retrieve recent news articles for NVDA to assess market sentiment and recent developments.", "status": "pending"}, {"content": "Analyze the data and news to identify trends, risks, and opportunities for NVDA.", "status": "pending"}, {"content": "Compile f

In [16]:
# This is the latest todo list
state["todos"]

[{'content': 'Identify key metrics and data points to research for NVDA stock (e.g., historical prices, financials, news, valuation metrics).',
  'status': 'completed'},
 {'content': 'Fetch historical financial data for NVDA using yfinance.',
  'status': 'completed'},
 {'content': 'Retrieve recent news articles for NVDA to assess market sentiment and recent developments.',
  'status': 'completed'},
 {'content': 'Analyze the data and news to identify trends, risks, and opportunities for NVDA.',
  'status': 'completed'},
 {'content': 'Compile findings into a structured research summary for NVDA stock.',
  'status': 'completed'}]

In [17]:
from IPython.display import Markdown

# Let's also have a look at the model response
Markdown(state["messages"][-1].content)

### **Nvidia (NVDA) Stock Research Summary**

---

#### **1. Overview of NVDA**
Nvidia (NVDA) is a global leader in **GPU (Graphics Processing Unit) technology**, **AI (Artificial Intelligence) computing**, and **graphics cards**. The company's dominance in the AI and data center markets has made it a key player in the tech sector. NVDA's stock is highly sensitive to trends in AI, gaming, and data center demand.

---

#### **2. Key Financial Metrics (as of latest data)**
| **Metric**               | **Value**                     | **Notes**                                  |
|--------------------------|-------------------------------|--------------------------------------------|
| **Current Price**        | $201.30                       | As of latest market close                 |
| **Market Cap**           | $4.88 trillion               | One of the largest in the world            |
| **P/E Ratio**            | 30.83                         | High valuation reflects growth expectations|
| **Forward P/E**          | 15.81                         | Lower forward P/E suggests positive outlook|
| **PEG Ratio**            | 0.64                          | Indicates potential undervaluation         |
| **Dividend Yield**       | 0.48%                         | Modest but growing dividends               |
| **Beta**                 | 2.20                          | High volatility compared to the market     |
| **52-Week High**         | $236.54                       | Recent peak                               |
| **52-Week Low**          | $145.50                       | Recent low                                |
| **50-Day Moving Average**| $209.81                       | Short-term trend                           |
| **200-Day Moving Average**| $190.09                      | Long-term trend                            |
| **Volume**               | 94.6M                         | High liquidity                            |

---

#### **3. Historical Price Analysis**
- **Trend**: NVDA has shown significant volatility, with a strong upward trend in 2024-2025, driven by AI adoption and demand for GPUs in data centers.
- **Recent Dip**: In **June 2026**, NVDA experienced a sharp decline (from ~$236 to ~$201) due to:
  - **Cooling AI trade**: Investors are pulling back from high-growth tech stocks.
  - **Semiconductor selloff**: Broader market weakness in semiconductor stocks.
  - **Profit-taking**: After a strong run-up, some investors are locking in gains.
- **Support Levels**: Key support levels are around **$190-$200**, while resistance is near **$236**.

---

#### **4. Recent News and Market Sentiment**
NVDA's stock has been influenced by the following recent developments:

1. **Tech Selloff (June 23, 2026)**
   - NVDA, along with other AI-related stocks (Micron, AMD), led a tech selloff as the **AI trade cooled**.
   - Concerns about **overvaluation** and **profit-taking** contributed to the decline.
   - **S&P 500 and Nasdaq** also fell due to semiconductor stock weakness.

2. **Nvidia's Network Ambitions**
   - Nvidia is expanding into **telecom and network infrastructure**, which could reshape AI infrastructure.
   - This long-term play could diversify revenue streams beyond GPUs.

3. **Competitive Landscape**
   - **AMD and Intel** are ramping up their AI and GPU offerings, increasing competition.
   - **ARM Holdings** and other chipmakers are also vying for market share in AI and mobile computing.

4. **Macro Environment**
   - **Federal Reserve policy**: Interest rate decisions impact high-growth stocks like NVDA.
   - **Geopolitical risks**: Supply chain disruptions and US-China tensions could affect production.

---

#### **5. Trends, Risks, and Opportunities**

##### **Opportunities**
- **AI and Data Center Growth**: NVDA dominates the AI GPU market, with demand expected to grow as AI adoption accelerates.
- **Expansion into New Markets**: Networking, automotive (self-driving cars), and robotics present long-term growth avenues.
- **Strong Financials**: High profitability, robust cash flow, and R&D investment ensure NVDA's leadership in innovation.

##### **Risks**
- **Valuation Concerns**: High P/E and PEG ratios suggest NVDA is priced for perfection. Any disappointment in earnings or growth could lead to a sharp correction.
- **Regulatory Risks**: Antitrust scrutiny and export restrictions (e.g., US-China trade tensions) could impact operations.
- **Competition**: AMD, Intel, and startups are investing heavily in AI and GPU technology.
- **Market Volatility**: NVDA's high beta (2.20) means it is more volatile than the broader market.

---

#### **6. Investment Recommendation**
| **Aspect**          | **Assessment**                                                                 |
|---------------------|-------------------------------------------------------------------------------|
| **Short-Term**      | **Neutral to Cautious** – Recent selloff may continue if the AI trade cools further. Watch for support at $190-$200. |
| **Long-Term**       | **Bullish** – NVDA remains a leader in AI and GPU technology, with strong fundamentals and growth potential. |
| **Risk Level**      | **High** – Due to volatility and valuation concerns.                          |
| **Suitable For**    | Investors with a **high risk tolerance** and a **long-term horizon**.         |

---

#### **7. Key Takeaways**
- NVDA is a **bellwether for the AI and semiconductor industries**.
- Recent weakness is due to **profit-taking and market rotation**, not fundamental deterioration.
- **Long-term investors** should consider accumulating shares on dips, while **short-term traders** should watch for a rebound or further downside.
- Monitor **earnings reports**, **AI adoption trends**, and **Federal Reserve policy** for future movements.

---
**Final Thought**: NVDA remains a **high-quality growth stock** with significant upside potential, but its high volatility and valuation require a disciplined approach.